In [11]:
import csv
import numpy as np

file_path = "/content/Online Retail.csv"

invoice_list = []
product_list = []
quantity_list = []
price_list = []
country_list = []

with open(file_path, "r", encoding="utf-8-sig", newline="") as file:
    reader = csv.DictReader(file)

    for row in reader:
        invoice_list.append(row["InvoiceNo"])
        product_list.append(row["StockCode"])
        quantity_list.append(row["Quantity"])
        price_list.append(row["UnitPrice"])
        country_list.append(row["Country"])

print("CSV loaded successfully!")
print("Total records:", len(invoice_list))
print("First 5 invoice numbers:", invoice_list[:5])
print("First 5 products:", product_list[:5])
print("First 5 quantities:", quantity_list[:5])
print("First 5 prices:", price_list[:5])
print("First 5 countries:", country_list[:5])

CSV loaded successfully!
Total records: 541909
First 5 invoice numbers: ['536365', '536365', '536365', '536365', '536365']
First 5 products: ['85123A', '71053', '84406B', '84029G', '84029E']
First 5 quantities: ['6', '6', '8', '6', '6']
First 5 prices: ['2.55', '3.39', '2.75', '3.39', '3.39']
First 5 countries: ['United Kingdom', 'United Kingdom', 'United Kingdom', 'United Kingdom', 'United Kingdom']


In [12]:
# Convert Python lists into NumPy arrays

invoice = np.array(invoice_list, dtype=str)
product = np.array(product_list, dtype=str)

quantity = np.array(quantity_list, dtype=float)
price = np.array(price_list, dtype=float)

country = np.array(country_list, dtype=str)

# Check array properties
print("Quantity shape:", quantity.shape)
print("Quantity dimensions:", quantity.ndim)
print("Quantity size:", quantity.size)
print("Quantity data type:", quantity.dtype)

print("\nPrice shape:", price.shape)
print("Price dimensions:", price.ndim)
print("Price size:", price.size)
print("Price data type:", price.dtype)

print("\nFirst 5 quantities:", quantity[:5])
print("First 5 prices:", price[:5])

Quantity shape: (541909,)
Quantity dimensions: 1
Quantity size: 541909
Quantity data type: float64

Price shape: (541909,)
Price dimensions: 1
Price size: 541909
Price data type: float64

First 5 quantities: [6. 6. 8. 6. 6.]
First 5 prices: [2.55 3.39 2.75 3.39 3.39]


In [13]:
# Check invalid numeric values
print("Invalid quantities:", np.sum(~np.isfinite(quantity)))
print("Invalid prices:", np.sum(~np.isfinite(price)))

# Check zero and negative values
print("Zero quantities:", np.sum(quantity == 0))
print("Negative quantities:", np.sum(quantity < 0))

print("Zero prices:", np.sum(price == 0))
print("Negative prices:", np.sum(price < 0))

# Check cancellation invoices
cancellation_mask = np.char.startswith(invoice, "C")
print("Cancellation invoices:", np.sum(cancellation_mask))

Invalid quantities: 0
Invalid prices: 0
Zero quantities: 0
Negative quantities: 10624
Zero prices: 2515
Negative prices: 2
Cancellation invoices: 9288


In [14]:
# Identify valid numeric values
finite_mask = np.isfinite(quantity) & np.isfinite(price)

# Identify valid positive sales
valid_sales_mask = (
    finite_mask
    & (quantity > 0)
    & (price > 0)
    & (~cancellation_mask)
)

# Filter the arrays using Boolean indexing
sales_invoice = invoice[valid_sales_mask]
sales_product = product[valid_sales_mask]
sales_quantity = quantity[valid_sales_mask]
sales_price = price[valid_sales_mask]
sales_country = country[valid_sales_mask]

print("Original records:", quantity.size)
print("Valid sales records:", sales_quantity.size)
print("Filtered records:", quantity.size - sales_quantity.size)

print("\nFirst 10 sales quantities:", sales_quantity[:10])
print("First 10 sales prices:", sales_price[:10])

Original records: 541909
Valid sales records: 530104
Filtered records: 11805

First 10 sales quantities: [ 6.  6.  8.  6.  6.  2.  6.  6.  6. 32.]
First 10 sales prices: [2.55 3.39 2.75 3.39 3.39 7.65 4.25 1.85 1.85 1.69]


In [15]:
# Calculate revenue for each sales transaction
revenue = sales_quantity * sales_price

# Descriptive statistics
print("Total Revenue: £", np.sum(revenue))
print("Average Revenue per Record: £", np.mean(revenue))
print("Median Revenue: £", np.median(revenue))
print("Minimum Revenue: £", np.min(revenue))
print("Maximum Revenue: £", np.max(revenue))
print("Revenue Standard Deviation: £", np.std(revenue))

print("\nFirst 10 revenue values:")
print(revenue[:10])

Total Revenue: £ 10666684.544
Average Revenue per Record: £ 20.121871451639677
Median Revenue: £ 9.9
Minimum Revenue: £ 0.001
Maximum Revenue: £ 168469.6
Revenue Standard Deviation: £ 270.3564882147936

First 10 revenue values:
[15.3  20.34 22.   20.34 20.34 15.3  25.5  11.1  11.1  54.08]


In [16]:
# Get unique product codes and their counts
unique_products, product_counts = np.unique(
    sales_product,
    return_counts=True
)

# Total quantity sold for each product
total_quantity_per_product = np.array([
    np.sum(sales_quantity[sales_product == p])
    for p in unique_products
])

# Sort products by total quantity sold, highest first
top_indices = np.argsort(total_quantity_per_product)[::-1][:10]

print("TOP 10 PRODUCTS BY QUANTITY SOLD")
print("-" * 45)

for rank, idx in enumerate(top_indices, start=1):
    print(
        rank,
        "| Product:", unique_products[idx],
        "| Quantity Sold:", total_quantity_per_product[idx],
        "| Transaction Rows:", product_counts[idx]
    )

TOP 10 PRODUCTS BY QUANTITY SOLD
---------------------------------------------
1 | Product: 23843 | Quantity Sold: 80995.0 | Transaction Rows: 1
2 | Product: 23166 | Quantity Sold: 78033.0 | Transaction Rows: 250
3 | Product: 22197 | Quantity Sold: 56921.0 | Transaction Rows: 1426
4 | Product: 84077 | Quantity Sold: 55047.0 | Transaction Rows: 537
5 | Product: 85099B | Quantity Sold: 48474.0 | Transaction Rows: 2112
6 | Product: 85123A | Quantity Sold: 37660.0 | Transaction Rows: 2265
7 | Product: 84879 | Quantity Sold: 36461.0 | Transaction Rows: 1489
8 | Product: 21212 | Quantity Sold: 36419.0 | Transaction Rows: 1370
9 | Product: 23084 | Quantity Sold: 30788.0 | Transaction Rows: 1036
10 | Product: 22492 | Quantity Sold: 26633.0 | Transaction Rows: 380


In [17]:
# Find unique countries
unique_countries = np.unique(sales_country)

# Calculate total revenue for each country
country_revenue = np.array([
    np.sum(revenue[sales_country == c])
    for c in unique_countries
])

# Sort countries by revenue (highest first)
top_country_indices = np.argsort(country_revenue)[::-1][:10]

print("TOP 10 COUNTRIES BY REVENUE")
print("-" * 50)

for rank, idx in enumerate(top_country_indices, start=1):
    print(
        rank,
        "| Country:", unique_countries[idx],
        "| Revenue: £", round(country_revenue[idx], 2)
    )

TOP 10 COUNTRIES BY REVENUE
--------------------------------------------------
1 | Country: United Kingdom | Revenue: £ 9025222.08
2 | Country: Netherlands | Revenue: £ 285446.34
3 | Country: EIRE | Revenue: £ 283453.96
4 | Country: Germany | Revenue: £ 228867.14
5 | Country: France | Revenue: £ 209715.11
6 | Country: Australia | Revenue: £ 138521.31
7 | Country: Spain | Revenue: £ 61577.11
8 | Country: Switzerland | Revenue: £ 57089.9
9 | Country: Belgium | Revenue: £ 41196.34
10 | Country: Sweden | Revenue: £ 38378.33


In [18]:
# Descriptive statistics of transaction revenue

print("===== REVENUE STATISTICS =====")

print("Total Revenue:", round(np.sum(revenue), 2))
print("Mean Revenue:", round(np.mean(revenue), 2))
print("Median Revenue:", round(np.median(revenue), 2))
print("Standard Deviation:", round(np.std(revenue), 2))
print("Variance:", round(np.var(revenue), 2))

print("\n===== MINIMUM AND MAXIMUM =====")

min_index = np.argmin(revenue)
max_index = np.argmax(revenue)

print("Minimum Revenue:", round(np.min(revenue), 2))
print("Maximum Revenue:", round(np.max(revenue), 2))

print("Minimum Revenue Product:", sales_product[min_index])
print("Maximum Revenue Product:", sales_product[max_index])

print("\n===== DATASET SUMMARY =====")

print("Total sales records:", revenue.size)
print("Unique products:", np.unique(sales_product).size)
print("Unique countries:", np.unique(sales_country).size)

===== REVENUE STATISTICS =====
Total Revenue: 10666684.54
Mean Revenue: 20.12
Median Revenue: 9.9
Standard Deviation: 270.36
Variance: 73092.63

===== MINIMUM AND MAXIMUM =====
Minimum Revenue: 0.0
Maximum Revenue: 168469.6
Minimum Revenue Product: PADS
Maximum Revenue Product: 23843

===== DATASET SUMMARY =====
Total sales records: 530104
Unique products: 3922
Unique countries: 38


In [19]:
# SMART RETAIL SALES INTELLIGENCE
# Final Business Insights Report

top_product_idx = np.argmax(total_quantity_per_product)
top_country_idx = np.argmax(country_revenue)

print("=" * 50)
print("       SMART RETAIL SALES INTELLIGENCE")
print("=" * 50)

print(f"\n1. Total Revenue: £{np.sum(revenue):,.2f}")

print(
    f"2. Average Revenue per Record: "
    f"£{np.mean(revenue):,.2f}"
)

print(
    f"3. Top-Selling Product: "
    f"{unique_products[top_product_idx]}"
)

print(
    f"   Total Quantity Sold: "
    f"{total_quantity_per_product[top_product_idx]:,.0f}"
)

print(
    f"4. Highest-Revenue Country: "
    f"{unique_countries[top_country_idx]}"
)

print(
    f"   Country Revenue: "
    f"£{country_revenue[top_country_idx]:,.2f}"
)

print(f"5. Unique Products Sold: {sales_product.size and np.unique(sales_product).size}")
print(f"6. Countries Covered: {np.unique(sales_country).size}")

print("\n" + "=" * 50)
print("Analysis completed using NumPy!")
print("=" * 50)

       SMART RETAIL SALES INTELLIGENCE

1. Total Revenue: £10,666,684.54
2. Average Revenue per Record: £20.12
3. Top-Selling Product: 23843
   Total Quantity Sold: 80,995
4. Highest-Revenue Country: United Kingdom
   Country Revenue: £9,025,222.08
5. Unique Products Sold: 3922
6. Countries Covered: 38

Analysis completed using NumPy!


In [20]:
# Create a 2D array using the first 12 revenue values
sample_revenue = revenue[:12].reshape(3, 4)

print("2D Revenue Array:")
print(sample_revenue)

print("\nShape:", sample_revenue.shape)
print("Dimensions:", sample_revenue.ndim)

# Descriptive methods along axes
print("\nColumn totals (axis=0):")
print(np.sum(sample_revenue, axis=0))

print("\nRow totals (axis=1):")
print(np.sum(sample_revenue, axis=1))

# Broadcasting: add a different adjustment to each column
adjustment = np.array([10, 20, 30, 40])
adjusted_revenue = sample_revenue + adjustment

print("\nBroadcasting result:")
print(adjusted_revenue)

2D Revenue Array:
[[15.3  20.34 22.   20.34]
 [20.34 15.3  25.5  11.1 ]
 [11.1  54.08 12.6  12.6 ]]

Shape: (3, 4)
Dimensions: 2

Column totals (axis=0):
[46.74 89.72 60.1  44.04]

Row totals (axis=1):
[77.98 72.24 90.38]

Broadcasting result:
[[25.3  40.34 52.   60.34]
 [30.34 35.3  55.5  51.1 ]
 [21.1  74.08 42.6  52.6 ]]


In [21]:
# Final project summary

summary = {
    "Total Revenue (£)": round(float(np.sum(revenue)), 2),
    "Average Revenue per Record (£)": round(float(np.mean(revenue)), 2),
    "Median Revenue (£)": round(float(np.median(revenue)), 2),
    "Total Sales Records": int(revenue.size),
    "Unique Products": int(np.unique(sales_product).size),
    "Countries Covered": int(np.unique(sales_country).size),
    "Top Product by Quantity": str(unique_products[np.argmax(total_quantity_per_product)]),
    "Top Country by Revenue": str(unique_countries[np.argmax(country_revenue)])
}

for metric, value in summary.items():
    print(f"{metric}: {value}")

Total Revenue (£): 10666684.54
Average Revenue per Record (£): 20.12
Median Revenue (£): 9.9
Total Sales Records: 530104
Unique Products: 3922
Countries Covered: 38
Top Product by Quantity: 23843
Top Country by Revenue: United Kingdom
